In [1]:
import json
import time
from typing import Any, Dict, List, Optional
import requests
from openai import OpenAI


class LLMClient:
    """OpenAI-compatible chat-completions client."""
    def __init__(self, base_url, api_key, timeout=60, max_retry=3, host=None, appid=None):
        self.base_url = base_url
        self.api_key = api_key
        self.max_retry = max_retry
        self.sleep_seconds = 3
        self.timeout = timeout
        
        default_headers = {}
        if host:
            default_headers["Host"] = host
        if appid:
            default_headers["appid"] = appid
            
        # 创建一个客户端实例
        client_kwargs = {
            "base_url": base_url,
            "api_key": api_key,
            "timeout": timeout,
            "max_retries": 0,  # 我们手动处理重试
        }
        
        if default_headers:
            client_kwargs["default_headers"] = default_headers
            
        try:
            self.client = OpenAI(**client_kwargs)
            print(f"OpenAI client initialized with base_url: {base_url}")
        except Exception as e:
            print(f"Failed to initialize OpenAI client: {e}")
            self.client = None

    def fetch_response(self,
                      query: str,
                      doc: str,
                      prompt: str = None,
                      messages: Optional[List[Dict[str, str]]] = None,
                      model: Optional[str] = None,
                      temperature: Optional[float] = None,
                      max_tokens: Optional[int] = None,
                      extra_body: Optional[Dict[str, Any]] = None,
                      ) -> Dict[str, Any]:
        
        # 如果既没有messages也没有prompt，抛出错误
        if messages is None and prompt is None:
            raise ValueError("Either 'messages' or 'prompt' must be provided")
            
        # 如果提供了prompt但没有messages，使用prompt构建messages
        if messages is None and prompt is not None:
            messages = [{"role": "user", "content": prompt}]
            
        # 构建请求参数
        request_params = {
            "model": model,
            "messages": messages,
            "temperature": temperature if temperature is not None else 0.8,
            "max_tokens": max_tokens if max_tokens is not None else 8196,
        }
        
        # 如果有 extra_body，添加到请求参数中
        # 注意：对于 OpenAI SDK，extra_body 是独立参数
        extra_params = {}
        if extra_body:
            extra_params = extra_body
            
        print("Request params:", json.dumps(request_params, ensure_ascii=False, indent=2))
        print("-" * 100)
        print()
        
        last_error = None
        
        # 重试逻辑
        for retry_idx in range(self.max_retry):
            try:
                # 检查客户端是否初始化成功
                if self.client is None:
                    # 如果 OpenAI 客户端未初始化，使用 requests 直接调用
                    return self._request_with_requests(
                        messages=messages,
                        model=model,
                        temperature=temperature,
                        max_tokens=max_tokens,
                        extra_body=extra_body
                    )
                
                # 使用 OpenAI SDK
                response = self.client.chat.completions.create(
                    **request_params,
                    **extra_params
                )
                
                # 检查响应类型
                if isinstance(response, str):
                    print(f"Warning: Received string response, attempting to parse as JSON")
                    try:
                        response_dict = json.loads(response)
                        return response_dict
                    except json.JSONDecodeError:
                        # 如果无法解析为JSON，尝试构造标准格式
                        return {
                            "choices": [
                                {
                                    "message": {
                                        "content": response
                                    }
                                }
                            ]
                        }
                
                # OpenAI SDK 返回的是对象，需要转成 dict
                if hasattr(response, 'model_dump'):
                    output_json = response.model_dump(mode="json")
                elif hasattr(response, 'dict'):
                    output_json = response.dict()
                else:
                    # 如果既没有 model_dump 也没有 dict，尝试直接转换为字典
                    output_json = response.__dict__ if hasattr(response, '__dict__') else {"choices": [{"message": {"content": str(response)}}]}
                    
                return output_json

            except Exception as e:
                last_error = e
                print(f"[Retry {retry_idx + 1}/{self.max_retry}] 请求失败: {e}")
                print(f"Error type: {type(e)}")
                if hasattr(e, '__dict__'):
                    print(f"Error details: {e.__dict__ if hasattr(e, '__dict__') else str(e)}")
                    
                if retry_idx < self.max_retry - 1:
                    time.sleep(self.sleep_seconds)
                else:
                    raise RuntimeError(f"请求重试 {self.max_retry} 次后仍失败: {last_error}")

    def _request_with_requests(self,
                              messages: List[Dict[str, str]],
                              model: Optional[str] = None,
                              temperature: Optional[float] = None,
                              max_tokens: Optional[int] = None,
                              extra_body: Optional[Dict[str, Any]] = None,
                              ) -> Dict[str, Any]:
        """使用 requests 直接调用 API（备用方法）"""
        
        url = self.base_url.rstrip("/")
        if not url.endswith("/chat/completions"):
            url = url + "/chat/completions"
        
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            if self.api_key.startswith("Bearer "):
                headers["Authorization"] = self.api_key
            else:
                headers["Authorization"] = f"Bearer {self.api_key}"
        
        payload = {
            "model": model,
            "messages": messages,
            "temperature": temperature if temperature is not None else 0.8,
            "max_tokens": max_tokens if max_tokens is not None else 8196,
        }
        
        if extra_body:
            payload.update(extra_body)
            
        print(f"Using requests to call: {url}")
        print(f"Payload: {json.dumps(payload, ensure_ascii=False, indent=2)}")
        
        for attempt in range(1, self.max_retry + 1):
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
                resp.raise_for_status()
                return resp.json()
            except Exception as e:
                print(f"[Attempt {attempt}/{self.max_retry}] Request failed: {e}")
                if attempt < self.max_retry:
                    time.sleep(self.sleep_seconds)
                else:
                    raise

    def parse_chat_content(self, response: Dict[str, Any]) -> str:
        """Extract assistant text from an OpenAI-compatible response."""
        try:
            # 尝试多种可能的响应格式
            if "choices" in response and len(response["choices"]) > 0:
                choice = response["choices"][0]
                if "message" in choice and "content" in choice["message"]:
                    return str(choice["message"]["content"])
                elif "text" in choice:
                    return str(choice["text"])
                elif "content" in choice:
                    return str(choice["content"])
            
            # 如果响应本身就是内容
            if isinstance(response, str):
                return response
            elif "content" in response:
                return str(response["content"])
            elif "response" in response:
                return str(response["response"])
            
            raise ValueError(f"无法从响应中提取内容: {response}")
        except (KeyError, IndexError, TypeError) as exc:
            raise ValueError(f"Invalid chat completion response: {response}") from exc

    def generate_text(self,
                     query: str,
                     doc: str,
                     prompt: str = None,
                     messages: Optional[List[Dict[str, str]]] = None,
                     model: Optional[str] = None,
                     temperature: Optional[float] = None,
                     max_tokens: Optional[int] = None,
                     extra_body: Optional[Dict[str, Any]] = None,
                     ) -> Dict[str, Any]:
        """Generate text using the LLM."""
        
        response = self.fetch_response(
            query=query,
            doc=doc,
            prompt=prompt,
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            extra_body=extra_body,
        )
        
        return {
            "query": query,
            "doc": doc,
            "request_data": self.parse_chat_content(response),
            "full_response": response,  # 保留完整响应以便调试
        }

    def request_model(
        self,
        *,
        prompt: Optional[str] = None,
        messages: Optional[List[Dict[str, str]]] = None,
        model: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        extra_body: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        """使用 requests 直接调用 OpenAI 兼容的 API（备用方法）"""
        
        if not self.base_url:
            raise RuntimeError("OPENAI_API_BASE is empty. Please set it first.")

        url = self.base_url.rstrip("/")
        if not url.endswith("/chat/completions"):
            url = url + "/chat/completions"

        headers = {"Content-Type": "application/json"}
        if self.api_key:
            if self.api_key.startswith("Bearer "):
                headers["Authorization"] = self.api_key
            else:
                headers["Authorization"] = f"Bearer {self.api_key}"

        if messages is None:
            messages = [{"role": "user", "content": prompt or ""}]

        payload: Dict[str, Any] = {
            "model": model,
            "messages": messages,
            "temperature": 0.2 if temperature is None else temperature,
        }
        if max_tokens is not None:
            payload["max_tokens"] = max_tokens
        if extra_body:
            payload.update(extra_body)

        last_error: Optional[Exception] = None
        for attempt in range(1, self.max_retry + 1):
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
                resp.raise_for_status()
                return resp.json()
            except Exception as exc:
                last_error = exc
                print(f"[Attempt {attempt}/{self.max_retry}] Request failed: {exc}")
                if attempt < self.max_retry:
                    time.sleep(self.sleep_seconds)

        raise RuntimeError(f"LLM request failed after {self.max_retry} retries: {last_error}")


if __name__ == "__main__":
    
    # 测试两种方式
    test_cases = [
        {
            "name": "使用 OpenAI SDK",
            "use_sdk": True,
        },
        {
            "name": "使用 requests（备用）",
            "use_sdk": False,
        }
    ]
    
    for test in test_cases:
        print(f"\n{'='*60}")
        print(f"测试: {test['name']}")
        print(f"{'='*60}\n")
        
        try:
            llmclient = LLMClient(
                base_url="http://10.11.175.3/tianchi/chat/completions",
                api_key="2cff86c63848008ab7982b5c63c9",
                timeout=60,
                max_retry=3,
                appid='app-CdjpA4YQ'
            )
            
            # 如果使用 OpenAI SDK 但初始化失败，会自动切换到 requests
            if not test['use_sdk']:
                # 强制使用 requests
                response = llmclient.request_model(
                    messages=[
                        {"role": "system", "content": "你是一个严谨、简洁的中文助手。"},
                        {"role": "user", "content": "你好，请简要介绍一下你自己。"}
                    ],
                    model="deepseek-v3.2",
                    temperature=0.2,
                    max_tokens=1024,
                    extra_body={"top_p": 0.9},
                )
            else:
                # 使用 generate_text（会尝试 OpenAI SDK）
                response = llmclient.generate_text(
                    query="你好",
                    doc="生成",
                    model="deepseek-v3.2",
                    messages=[
                        {"role": "system", "content": "你是一个严谨、简洁的中文助手。"},
                        {"role": "user", "content": "你好，请简要介绍一下你自己。"}
                    ],
                    temperature=0.2,
                    max_tokens=1024,
                    extra_body={"top_p": 0.9},
                )

            print("✓ 响应成功!")
            print("\nCONTENT:")
            print(response["request_data"])
            print(f"\n完整响应结构: {list(response.keys())}")
            
        except Exception as e:
            print(f"✗ 测试失败: {e}")
            import traceback
            traceback.print_exc()


测试: 使用 OpenAI SDK

OpenAI client initialized with base_url: http://10.11.175.3/tianchi/chat/completions
Request params: {
  "model": "deepseek-v3.2",
  "messages": [
    {
      "role": "system",
      "content": "你是一个严谨、简洁的中文助手。"
    },
    {
      "role": "user",
      "content": "你好，请简要介绍一下你自己。"
    }
  ],
  "temperature": 0.2,
  "max_tokens": 1024
}
----------------------------------------------------------------------------------------------------

✓ 响应成功!

CONTENT:
<!DOCTYPE html>              <html>   <!--STATUS OK-->  <head>   <meta content="text/html; charset=utf-8" http-equiv='Content-Type'/>  <title>页面不存在_百度搜索</title> <link rel="shortcut icon" href="//img1.bdstatic.com/static/common/img/icon_8a1e2b4.png" type="image/x-icon"> <link rel="icon" sizes="any" href="//img1.bdstatic.com/static/common/img/icon_8a1e2b4.png">  <script>
    void function(a,b,c,d,e,f,g){a.alogObjectName=e,a[e]=a[e]||function(){(a[e].q=a[e].q||[]).push(arguments)},a[e].l=a[e].l||+new Date,d="https:"===a.

Traceback (most recent call last):
  File "<ipython-input-1-f61351bf3b2a>", line 334, in <module>
    extra_body={"top_p": 0.9},
  File "<ipython-input-1-f61351bf3b2a>", line 292, in request_model
    raise RuntimeError(f"LLM request failed after {self.max_retry} retries: {last_error}")
RuntimeError: LLM request failed after 3 retries: Expecting value: line 1 column 1 (char 0)
